In [2]:
# Core libraries
import pandas as pd
import numpy as np

In [3]:
# Path handling (OS-independent)
from pathlib import Path

In [4]:
# Project root directory (repo root)
PROJECT_ROOT = Path.cwd().parent

# Data and output directories
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Sanity check
print("Project root:", PROJECT_ROOT)
print("Data directory exists:", DATA_DIR.exists())
print("Output directory exists:", OUTPUT_DIR.exists())


Project root: c:\Users\NIVAS\Desktop\Jobaaj\ecommerce-sql-python-analysis
Data directory exists: True
Output directory exists: True


## Data Loading (OS-Independent)

All datasets are loaded using relative paths to ensure the notebook runs
on any machine after cloning the GitHub repository.

In [5]:
# File paths (relative to project root)
customers_path = DATA_DIR / "customers.xlsx"
orders_path = DATA_DIR / "orders.xlsx"
order_items_path = DATA_DIR / "order_items.xlsx"
payments_path = DATA_DIR / "payments.xlsx"
products_path = DATA_DIR / "products.xlsx"
sellers_path = DATA_DIR / "sellers.xlsx"
geolocation_path = DATA_DIR / "geolocation.csv"

In [6]:
# Load datasets
customers = pd.read_excel(customers_path)
orders = pd.read_excel(orders_path)
order_items = pd.read_excel(order_items_path)
payments = pd.read_excel(payments_path)
products = pd.read_excel(products_path)
sellers = pd.read_excel(sellers_path)
geolocation = pd.read_csv(geolocation_path)

In [7]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation
}

for name, df in datasets.items():
    print(f"{name.upper():15} | Rows: {df.shape[0]:6} | Columns: {df.shape[1]}")

CUSTOMERS       | Rows:  99441 | Columns: 5
ORDERS          | Rows:  99441 | Columns: 8
ORDER_ITEMS     | Rows: 112650 | Columns: 7
PAYMENTS        | Rows: 103886 | Columns: 5
PRODUCTS        | Rows:  32951 | Columns: 9
SELLERS         | Rows:   3095 | Columns: 4
GEOLOCATION     | Rows: 1000163 | Columns: 5


In [8]:
# Ensure output directory exists
OUTPUT_DIR.mkdir(exist_ok=True)

In [9]:
# Export cleaned datasets to CSV for MySQL ingestion
customers.to_csv(OUTPUT_DIR / "customers.csv", index=False)
orders.to_csv(OUTPUT_DIR / "orders.csv", index=False)
order_items.to_csv(OUTPUT_DIR / "order_items.csv", index=False)
payments.to_csv(OUTPUT_DIR / "payments.csv", index=False)
products.to_csv(OUTPUT_DIR / "products.csv", index=False)
sellers.to_csv(OUTPUT_DIR / "sellers.csv", index=False)
geolocation.to_csv(OUTPUT_DIR / "geolocation.csv", index=False)

In [10]:
# Verify exported files
for file in OUTPUT_DIR.iterdir():
    print(file.name)

customers.csv
geolocation.csv
monthly_revenue.csv
orders.csv
order_items.csv
order_level_data.csv
payments.csv
products.csv
product_price_vs_purchase.csv
sellers.csv


# BASIC PROBLEMS

### Problem 1: List all unique cities where customers are located.

In [56]:
unique_cities_df = (
    customers[['customer_city']]
    .dropna()
    .drop_duplicates()
    .sort_values(by='customer_city')
    .reset_index(drop=True)
)

display(unique_cities_df)

print("Total unique cities:", unique_cities_df.shape[0])

,customer_city
0,abadia dos dourados
1,abadiania
2,abaete
3,abaetetuba
4,abaiara
...,...
4114,xinguara
4115,xique-xique
4116,zacarias
4117,ze doca


Total unique cities: 4119


### Problem 2: Count the number of orders placed in 2017.

In [57]:
# Ensure timestamp column is datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Filter orders from 2017
orders_2017 = orders[orders['order_purchase_timestamp'].dt.year == 2017]

# Count distinct orders
total_orders_2017 = orders_2017['order_id'].nunique()

print("Total orders in 2017:", total_orders_2017)

Total orders in 2017: 45101


### Problem 3: Find the total sales per category.

In [58]:
# Merge using correct column name
category_sales = (
    order_items
    .merge(products[['product_id', 'product category']],
           on='product_id',
           how='left')
    .groupby('product category')['price']
    .sum()
    .reset_index()
    .sort_values(by='price', ascending=False)
)

# Rename for clarity
category_sales.rename(columns={
    'product category': 'product_category',
    'price': 'total_sales'
}, inplace=True)

display(category_sales)

# Overall revenue
overall_revenue = order_items['price'].sum()
print("\nOverall revenue:", round(overall_revenue, 2))


,product_category,total_sales
30,HEALTH BEAUTY,1258681.34
45,Watches present,1205005.68
49,bed table bath,1036988.68
68,sport leisure,988048.97
53,computer accessories,911954.32
...,...,...
58,flowers,1110.04
32,House Comfort 2,760.27
50,cds music dvds,730.00
18,Fashion Children's Clothing,569.85



Overall revenue: 13591643.7


### Problem 4: Calculate the percentage of orders that were paid in installments.

In [59]:
# Total distinct orders
total_orders = orders['order_id'].nunique()

# Orders paid in installments (>1)
installment_orders = (
    payments[payments['payment_installments'] > 1]['order_id']
    .nunique()
)

# Calculate percentage
installment_percentage = round(
    (installment_orders / total_orders) * 100,
    2
)

print("Total Orders:", total_orders)
print("Installment Orders:", installment_orders)
print("Installment Percentage:", installment_percentage, "%")


Total Orders: 99441
Installment Orders: 51170
Installment Percentage: 51.46 %


### Problem 5: Calculate the percentage of orders that were paid in installments.

In [60]:
customers_by_state = (
    customers
    .groupby('customer_state')['customer_unique_id']
    .nunique()
    .reset_index()
    .sort_values(by='customer_unique_id', ascending=False)
)

customers_by_state.rename(
    columns={'customer_unique_id': 'total_customers'},
    inplace=True
)

display(customers_by_state)

,customer_state,total_customers
25,SP,40302
18,RJ,12384
10,MG,11259
22,RS,5277
17,PR,4882
23,SC,3534
4,BA,3277
6,DF,2075
7,ES,1964
8,GO,1952


# INTERMEDIATE PROBLEMS

### Problem 1: Calculate the number of orders per month in 2018.

In [62]:
# Filter 2018 orders
orders_2018 = orders[orders['order_purchase_timestamp'].dt.year == 2018]

# Group by month
monthly_orders_2018 = (
    orders_2018
    .groupby(orders_2018['order_purchase_timestamp'].dt.month)['order_id']
    .nunique()
    .reset_index()
)

monthly_orders_2018.columns = ['order_month', 'total_orders']

display(monthly_orders_2018.sort_values('order_month'))

,order_month,total_orders
0,1,7269
1,2,6728
2,3,7211
3,4,6939
4,5,6873
5,6,6167
6,7,6292
7,8,6512
8,9,16
9,10,4


In [63]:
orders_2018['order_purchase_timestamp'].min()

Timestamp('2018-01-01 02:48:41')

In [64]:
orders_2018['order_purchase_timestamp'].max()

Timestamp('2018-10-17 17:30:18')

### Problem 2: Find the average number of products per order, grouped by customer city.

In [65]:
# Step 1: Count items per order
order_item_counts = (
    order_items
    .groupby('order_id')['order_item_id']
    .count()
    .reset_index()
    .rename(columns={'order_item_id': 'order_item_count'})
)

# Step 2: Merge with orders to get customer_id
order_with_customer = order_item_counts.merge(
    orders[['order_id', 'customer_id']],
    on='order_id',
    how='left'
)

# Step 3: Merge with customers to get city
order_with_city = order_with_customer.merge(
    customers[['customer_id', 'customer_city']],
    on='customer_id',
    how='left'
)

# Step 4: Calculate average per city
avg_products_per_city = (
    order_with_city
    .groupby('customer_city')['order_item_count']
    .mean()
    .reset_index()
    .rename(columns={'order_item_count': 'avg_products_per_order'})
    .sort_values(by='avg_products_per_order', ascending=False)
)

display(avg_products_per_city)

,customer_city,avg_products_per_order
2619,padre carvalho,7.0
907,celso ramos,6.5
756,candido godoi,6.0
1154,datas,6.0
2264,matias olimpio,5.0
...,...,...
1662,indiana,1.0
1663,indianopolis,1.0
1664,indiapora,1.0
1666,indiaroba,1.0


### Problem 3: Calculate the percentage of total revenue contributed by each product category.

In [66]:
# Merge datasets
category_revenue = (
    order_items
    .merge(products[['product_id', 'product category']],
           on='product_id',
           how='left')
    .groupby('product category')['price']
    .sum()
    .reset_index()
)

# Total revenue
total_revenue = order_items['price'].sum()

# Calculate percentage
category_revenue['revenue_percentage'] = round(
    (category_revenue['price'] / total_revenue) * 100,
    2
)

# Rename columns
category_revenue.rename(columns={
    'product category': 'product_category',
    'price': 'category_revenue'
}, inplace=True)

category_revenue = category_revenue.sort_values(
    by='revenue_percentage',
    ascending=False
)

display(category_revenue)

,product_category,category_revenue,revenue_percentage
30,HEALTH BEAUTY,1258681.34,9.26
45,Watches present,1205005.68,8.87
49,bed table bath,1036988.68,7.63
68,sport leisure,988048.97,7.27
53,computer accessories,911954.32,6.71
...,...,...,...
58,flowers,1110.04,0.01
32,House Comfort 2,760.27,0.01
2,Arts and Crafts,1814.01,0.01
62,insurance and services,283.29,0.00


### Problem 4: Identify the correlation between product price and the number of times a product has been purchased.

In [67]:
# Step 1: Product-level aggregation
product_analysis = (
    order_items
    .groupby('product_id')
    .agg(
        times_purchased=('order_item_id', 'count'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

# Step 2: Calculate correlation
correlation = product_analysis['avg_price'].corr(
    product_analysis['times_purchased']
)

print("Correlation between price and times purchased:", round(correlation, 4))

# Optional: View dataset
display(product_analysis.head())

Correlation between price and times purchased: -0.0321


,product_id,times_purchased,avg_price
0,00066f42aeeb9f3007548bb9d3f33c38,1,101.65
1,00088930e925c41fd95ebfe695fd2655,1,129.90
2,0009406fd7479715e4bef61dd91f2462,1,229.00
3,000b8f95fcb9e0096488278317764d19,2,58.90
4,000d9be29b5207b54e86aa1b1ac54872,1,199.00


In [68]:
# Improved product-level analysis with category

product_analysis = (
    order_items
    .merge(products[['product_id', 'product category']],
           on='product_id',
           how='left')
    .groupby(['product_id', 'product category'])
    .agg(
        times_purchased=('order_item_id', 'count'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

# Correlation
correlation = product_analysis['avg_price'].corr(
    product_analysis['times_purchased']
)

print("Correlation between price and times purchased:", round(correlation, 4))

display(product_analysis.head())


Correlation between price and times purchased: -0.0328


,product_id,product category,times_purchased,avg_price
0,00066f42aeeb9f3007548bb9d3f33c38,perfumery,1,101.65
1,00088930e925c41fd95ebfe695fd2655,automotive,1,129.90
2,0009406fd7479715e4bef61dd91f2462,bed table bath,1,229.00
3,000b8f95fcb9e0096488278317764d19,housewares,2,58.90
4,000d9be29b5207b54e86aa1b1ac54872,Watches present,1,199.00


### Problem 5: Calculate the total revenue generated by each seller, and rank them by revenue.

In [69]:
seller_revenue = (
    order_items
    .groupby('seller_id')
    .agg(total_revenue=('price', 'sum'),
         total_freight=('freight_value', 'sum'))
    .reset_index()
)

# Combine price + freight
seller_revenue['total_revenue'] = (
    seller_revenue['total_revenue'] +
    seller_revenue['total_freight']
)

# Keep only necessary columns
seller_revenue = seller_revenue[['seller_id', 'total_revenue']]

# Round values
seller_revenue['total_revenue'] = seller_revenue['total_revenue'].round(2)

# Ranking
seller_revenue['revenue_rank'] = (
    seller_revenue['total_revenue']
    .rank(method='dense', ascending=False)
)

# Sort
seller_revenue = seller_revenue.sort_values('revenue_rank')

display(seller_revenue.head(10))

,seller_id,total_revenue,revenue_rank
857,4869f7a5dfa277a7dca6462dcf3b52b2,249640.70,1.0
1535,7c67e1448b00f6e969d365cea6b010ab,239536.44,2.0
1013,53243585a1d6dc2643021fd1853d8905,235856.68,3.0
881,4a3ca9315b744ce9f8e9374361493884,235539.96,4.0
3024,fa1c13f2614d7b5c4749cbc52fecda94,204084.73,5.0
2643,da8622b14eb17ae2831f4ac5b9dab84a,185192.32,6.0
1560,7e93a43ef30c4f03f38b393420bc753a,182754.05,7.0
192,1025f0e2d44d7041d6cf58b6550e0bfa,172860.69,8.0
1505,7a67c85e85bb2ce8582c35f2203ad736,162648.38,9.0
1824,955fee9216a65b617aa5c0531780ce60,160602.68,10.0


# ADVANCE PROBLEMS

### Problem 1: Calculate the moving average of order values for each customer over their order history.

In [70]:
# Step 1: Calculate order value
order_values = (
    order_items
    .groupby('order_id')
    .agg(order_value=('price', 'sum'),
         freight=('freight_value', 'sum'))
    .reset_index()
)

order_values['order_value'] = order_values['order_value'] + order_values['freight']
order_values = order_values[['order_id', 'order_value']]

# Step 2: Merge with orders table
order_values = order_values.merge(
    orders[['order_id', 'customer_id', 'order_purchase_timestamp']],
    on='order_id',
    how='left'
)

# Ensure datetime
order_values['order_purchase_timestamp'] = pd.to_datetime(
    order_values['order_purchase_timestamp']
)

# Step 3: Sort
order_values = order_values.sort_values(
    ['customer_id', 'order_purchase_timestamp']
)

# Step 4: Calculate moving average (last 3 orders)
order_values['moving_avg_order_value'] = (
    order_values
    .groupby('customer_id')['order_value']
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

order_values['moving_avg_order_value'] = order_values['moving_avg_order_value'].round(2)

display(order_values.head(15))

,order_id,order_value,customer_id,order_purchase_timestamp,moving_avg_order_value
36660,5f79b5b0931d63f1a42989eb65b9da6e,114.74,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74
63005,a44895d095d7e0702b6a162fa2dbeced,67.41,000161a058600d5901f007fab4c27140,2017-07-16 09:40:32,67.41
18934,316a104623542e4d75189bb372bc5f8d,195.42,0001fd6190edaaf884bcaf3d49edf079,2017-02-28 11:06:43,195.42
33936,5825ce2e88d5346438686b0bba99e5ee,179.35,0002414f95344307404f0ace7a26f1d5,2017-08-16 13:09:20,179.35
4167,0ab7fb08086d4af9141453c91878ed7a,107.01,000379cdec625522490c315e70c7a9fb,2018-04-02 13:42:17,107.01
79083,cd3558a10d854487b4f907e9b326a4fc,71.80,0004164d20a9e969af783496f3408652,2017-04-12 08:35:12,71.80
3090,07f6c3baf9ac86865b60f640c4f923c6,49.40,000419c5494106c306a97b5635748086,2018-03-02 17:47:40,49.40
53821,8c3d752c5c02227878fae49aeaddbfd7,166.59,00046a560d407e99b969756e0b10f282,2017-12-18 11:08:30,166.59
96575,fa906f338cee30a984d0945b3832e431,85.23,00050bf6e01e69d5c0fd612f1bcfb69c,2017-09-17 16:04:44,85.23
59578,9b961b894e797f63622137ff7eb1c1af,1255.71,000598caf2ef4117407665ac33275130,2018-08-11 12:14:35,1255.71


### Problem 3: Calculate the cumulative sales per month for each year.

In [72]:
# Merge order value
monthly_data = (
    order_items
    .merge(orders[['order_id', 'order_purchase_timestamp']],
           on='order_id',
           how='left')
)

monthly_data['sales_year'] = monthly_data['order_purchase_timestamp'].dt.year
monthly_data['sales_month'] = monthly_data['order_purchase_timestamp'].dt.month

# Monthly revenue
monthly_sales = (
    monthly_data
    .groupby(['sales_year', 'sales_month'])
    .agg(monthly_sales=('price', 'sum'),
         freight=('freight_value', 'sum'))
    .reset_index()
)

monthly_sales['monthly_sales'] = (
    monthly_sales['monthly_sales'] + monthly_sales['freight']
)

monthly_sales = monthly_sales[['sales_year', 'sales_month', 'monthly_sales']]

# Sort
monthly_sales = monthly_sales.sort_values(['sales_year', 'sales_month'])

# Cumulative calculation
monthly_sales['cumulative_sales'] = (
    monthly_sales
    .groupby('sales_year')['monthly_sales']
    .cumsum()
    .round(2)
)

display(monthly_sales)

,sales_year,sales_month,monthly_sales,cumulative_sales
0,2016,9,354.75,354.75
1,2016,10,56808.84,57163.59
2,2016,12,19.62,57183.21
3,2017,1,137188.49,137188.49
4,2017,2,286280.62,423469.11
5,2017,3,432048.59,855517.70
6,2017,4,412422.24,1267939.94
7,2017,5,586190.95,1854130.89
8,2017,6,502963.04,2357093.93
9,2017,7,584971.62,2942065.55


### Problem 3: Calculate the year-over-year growth rate of total sales.

In [74]:
# Merge orders + order_items
merged = orders.merge(order_items, on='order_id')

# Extract year
merged['sales_year'] = merged['order_purchase_timestamp'].dt.year

# Calculate yearly total revenue
yearly_sales = (
    merged
    .groupby('sales_year')
    .apply(lambda x: (x['price'] + x['freight_value']).sum())
    .reset_index(name='total_sales')
)

# Calculate YoY growth
yearly_sales['yoy_growth_percentage'] = (
    yearly_sales['total_sales'].pct_change() * 100
).round(2)

display(yearly_sales)

C:\Users\NIVAS\AppData\Local\Temp\ipykernel_21164\3500651982.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x['price'] + x['freight_value']).sum())


,sales_year,total_sales,yoy_growth_percentage
0,2016,57183.21,NaN
1,2017,7142672.43,12390.86
2,2018,8643697.60,21.01


### Problem 4: Calculate the retention rate of customers, defined as the percentage of customers who make another purchase within 6 months of their first purchase.

In [54]:
# Step 1: First purchase per customer
first_purchase = (
    orders
    .groupby('customer_id')['order_purchase_timestamp']
    .min()
    .reset_index()
    .rename(columns={'order_purchase_timestamp': 'first_purchase_date'})
)

# Merge back to orders
merged_orders = orders.merge(first_purchase, on='customer_id')

# Step 2: Identify repeat purchases within 6 months
retained_customers = merged_orders[
    (merged_orders['order_purchase_timestamp'] > merged_orders['first_purchase_date']) &
    (merged_orders['order_purchase_timestamp'] <= 
     merged_orders['first_purchase_date'] + pd.DateOffset(months=6))
]['customer_id'].nunique()

# Step 3: Total customers
total_customers = orders['customer_id'].nunique()

# Step 4: Retention Rate
retention_rate = round((retained_customers / total_customers) * 100, 2)

print("Total Customers:", total_customers)
print("Retained Customers (within 6 months):", retained_customers)
print("Retention Rate:", retention_rate, "%")


Total Customers: 99441
Retained Customers (within 6 months): 0
Retention Rate: 0.0 %


### Problem 5: Identify the top 3 customers who spent the most money in each year.

In [55]:
# Merge tables
merged = orders.merge(order_items, on='order_id')

# Extract year
merged['sales_year'] = merged['order_purchase_timestamp'].dt.year

# Calculate total spent per customer per year
customer_yearly_spend = (
    merged
    .groupby(['sales_year', 'customer_id'])
    .apply(lambda x: (x['price'] + x['freight_value']).sum())
    .reset_index(name='total_spent')
)

# Rank within each year
customer_yearly_spend['spend_rank'] = (
    customer_yearly_spend
    .groupby('sales_year')['total_spent']
    .rank(method='dense', ascending=False)
)

# Filter top 3
top_3_customers = customer_yearly_spend[
    customer_yearly_spend['spend_rank'] <= 3
].sort_values(['sales_year', 'spend_rank'])

display(top_3_customers)


C:\Users\NIVAS\AppData\Local\Temp\ipykernel_21164\236393429.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x['price'] + x['freight_value']).sum())


,sales_year,customer_id,total_spent,spend_rank
213,2016,a9dc96b027d1252bbac0a9b72d837fc6,1423.55,1.0
37,2016,1d34ed25963d5aae4cf3d7f3a4cda173,1400.74,2.0
80,2016,4a06381959b6670756de02e07b83815f,1227.78,3.0
4156,2017,1617b1357756262bfa56ab541c47bc16,13664.08,1.0
35059,2017,c6e2731c5b391845f6800c97401a43a9,6929.31,2.0
11418,2017,3fd6777bbce08a352fddd04e4a7cc8f6,6726.66,3.0
94592,2018,ec5b2ba62e574342386871631fafd3fc,7274.88,1.0
96320,2018,f48d464a0baaea338cb25f816991ab1f,6922.21,2.0
92125,2018,e0a2412720e9ea4f26c1ac985f6a7358,4809.44,3.0


# END